In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
import shap
from matplotlib import pyplot as plt

In [3]:
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=12)

In [12]:
def plot_feature_importances(X_train, y_train, X_test, y_test, color_scheme = "green"):

    # Fit rf regressor to training data
    rf = RandomForestRegressor()
    rf.fit(X_train, y_train)

    # Fit permutation feature importance
    perm_importance = permutation_importance(rf, X_test, y_test)

    # Fit SHAP feature importance
    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_test)

    # Extract importance values from each method
    rf_importance_values = rf.feature_importances_
    perm_importance_values = perm_importance.importances_mean
    shap_importance_values = np.mean(np.abs(shap_values), axis=0)

    feature_names = X_train.columns if hasattr(X_train, 'columns') else [f"Feature {i}" for i in range(X_train.shape[1])]
    
    # Bundle the data together for iterative plotting
    methods = [
        ("MDI (Built-in RF)", rf_importance_values),
        ("Permutation Importance", perm_importance_values),
        ("SHAP (Global Absolute)", shap_importance_values)
    ]

    # Create the 1x3 subplot figure
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
    
    for i, (title, values) in enumerate(methods):
        ax = axes[i]
        
        # Sort indices in ascending order for a clean horizontal bar plot layout
        sorted_idx = np.argsort(values)
        
        # Plot horizontal bars
        ax.barh(np.array(feature_names)[sorted_idx], values[sorted_idx], color=color_scheme, edgecolor='black', alpha=0.8)
        
        # Style adjustments
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.set_xlabel('Importance Score', fontsize=12)
        ax.grid(axis='x', linestyle='--', alpha=0.5)
        
    plt.tight_layout()
    plt.show()

In [ ]:
plot_feature_importances(X_train, y_train, X_test, y_test)